In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import subprocess
from pathlib import Path

def extract_frames_ffmpeg(video_path, out_dir, fps):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_pattern = str(out_dir / "frame_%02d.jpg")

    subprocess.run(["ffmpeg","-y","-i", str(video_path), "-vf", f"fps={fps}", out_pattern])

videos_dir  = Path("/content/drive/MyDrive/job_recommendation_dataset/videos")
frames_root = Path("/content/drive/MyDrive/job_recommendation_dataset/frames")

for video_path in videos_dir.glob("*.mp4"):
    extract_frames_ffmpeg(video_path, frames_root / video_path.stem, fps=0.5)

In [3]:
from tensorflow.keras.applications.vgg16 import VGG16

vgg = VGG16(weights="imagenet", include_top=False, pooling="avg")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Sequential
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input

base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224,224,3))
base_model.trainable = False

In [5]:
feature_model = Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(256, activation="relu", name="compress_1"),

    layers.Dropout(0.3, name="dropout_1"),

    layers.Dense(128, activation=None, name="feat128")
])

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import matplotlib.pyplot as plt

frames_root = Path("/content/drive/MyDrive/job_recommendation_dataset/frames")

def video_folder_to_vector(video_folder, model, max_frames=60):
    img_paths = sorted(Path(video_folder).glob("*.jpg"))[:max_frames]

    batch = np.array(
        [img_to_array(load_img(p, target_size=(224, 224))) for p in img_paths],
        dtype=np.float32
    )

    batch = preprocess_input(batch)

    feats = model.predict(batch, verbose=0)

    return feats.mean(axis=0)

In [7]:
rows = []
video_folders = sorted([d for d in frames_root.iterdir() if d.is_dir()])
total_videos = len(video_folders)

for idx, video_folder in enumerate(video_folders, 1):
    video_id = video_folder.name

    vec = video_folder_to_vector(video_folder, feature_model, max_frames=60)

    row = {"video_id": video_id}
    for i, val in enumerate(vec):
        row[f"feat_{i}"] = float(val)
    rows.append(row)

df = pd.DataFrame(rows)

In [10]:
df.to_csv("/content/drive/MyDrive/job_recommendation_dataset/FrameExtractedFeatures.csv", index=False)

In [11]:
df

,video_id,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,...,feat_118,feat_119,feat_120,feat_121,feat_122,feat_123,feat_124,feat_125,feat_126,feat_127
0,1,2.350749,1.465922,0.108799,2.055667,-1.790537,-1.257199,-0.571225,0.675232,-2.142967,...,0.364913,2.491852,2.149050,-0.656319,0.410853,0.474060,0.558089,-2.846429,0.310791,0.777409
1,10,2.717313,0.225967,-2.066506,3.010669,-0.926412,-2.086051,-2.426939,0.286064,-0.912288,...,0.717903,2.483006,3.728958,0.171573,0.624234,3.369594,2.723764,-3.000538,-0.381222,-0.989059
2,11,4.559945,0.476071,-1.162536,4.560372,-2.387441,0.277245,-1.820904,-2.944745,0.397119,...,-1.310992,4.228568,3.364523,-0.713512,-1.163601,4.396499,2.366346,-3.502195,-2.406260,2.185118
3,12,3.713580,-0.617767,-0.306062,1.111537,-1.718939,-1.663195,-0.870850,0.306593,0.483555,...,0.520402,1.908455,0.513166,-0.138854,-0.753118,3.291436,1.115083,-1.135928,-2.331029,-0.349624
4,13,2.340033,1.673487,-0.598603,4.110760,-2.954622,-1.984098,-1.884868,-1.691133,-1.381578,...,-0.426767,1.183965,0.754517,0.247111,-1.852180,-0.334902,3.041859,-2.678728,-2.642048,-0.926759
5,14,7.968719,2.824521,2.277617,5.784925,-1.267398,-3.642244,-0.690088,-2.812609,-2.071015,...,4.550122,4.779362,0.291666,-4.597291,-3.539374,-0.370163,2.452378,-5.307971,-2.545341,0.020085
6,15,0.754062,-1.576007,1.488413,2.284607,-0.215401,-1.019619,-1.679425,-1.753798,-1.453651,...,-0.435651,1.664368,-0.726743,-0.787544,-0.673325,5.370200,3.624615,-3.718412,-2.309597,1.220442
7,16,3.567564,-0.985898,-0.428273,1.412196,-1.193037,-0.500380,-0.980231,-0.809071,-6.736492,...,-1.869798,4.051077,-1.684072,-1.629514,0.301603,3.212594,-1.248753,-5.167887,-1.435405,0.817377
8,17,2.399540,5.691093,-0.740847,0.541362,-3.815942,1.422678,-4.461206,-2.174037,-2.441392,...,1.837234,3.305154,0.269991,1.376981,-0.906222,2.759752,2.331106,-2.771579,1.980281,1.330389
9,18,3.764226,0.603946,2.849691,0.420501,-2.377357,1.003385,-3.910248,-1.210310,-5.087905,...,0.405518,0.278630,-0.075740,0.230959,1.449106,2.245022,-0.995920,-4.565494,-3.982214,4.029389
